In [1]:
# ==============================================================================
# CELL 1 — Setup Environment & Mount Drive
# ==============================================================================
!pip install -q transformers datasets evaluate accelerate

from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_DIR    = '/content/drive/MyDrive/Smart_Scan/Recognition_Model'
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'trocr_checkpoints')
FINAL_MODEL_DIR= os.path.join(PROJECT_DIR, 'trocr_final')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

print(f"[OK] Environment ready. Checkpoints -> {CHECKPOINT_DIR}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.0 MB/s eta 0:00:00
Mounted at /content/drive
[OK] Environment ready. Checkpoints -> /content/drive/MyDrive/Smart_Scan/Recognition_Model/trocr_checkpoints


In [2]:
# ==============================================================================
# CELL 2 — Load Dataset (Drive cache = instant re-runs, skips download)
# ==============================================================================
from datasets import load_dataset, load_from_disk
from transformers import TrOCRProcessor
import os
import shutil

TRAIN_CACHE = os.path.join(PROJECT_DIR, 'cache_train_dataset')
EVAL_CACHE  = os.path.join(PROJECT_DIR, 'cache_eval_dataset')

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-small-printed')

def preprocess_data(examples):
    pixel_values = processor(
        examples['image'], return_tensors='pt'
    ).pixel_values
    labels = processor.tokenizer(
        examples['formula'],
        padding='max_length',
        max_length=64,
        truncation=True        # FIX: truncate sequences > 64 tokens
    ).input_ids
    labels = [
        [tok if tok != processor.tokenizer.pad_token_id else -100 for tok in seq]
        for seq in labels
    ]
    return {'pixel_values': pixel_values.squeeze(), 'labels': labels}

# -------------------------------------------------------------------
# FAST PATH: load from Drive cache (all re-runs after first)
# -------------------------------------------------------------------
cache_loaded = False
if os.path.exists(TRAIN_CACHE) and os.path.exists(EVAL_CACHE):
    print('[CACHE HIT] Loading processed datasets from Drive...')
    try:
        train_dataset = load_from_disk(TRAIN_CACHE)
        eval_dataset  = load_from_disk(EVAL_CACHE)
        print(f'[OK] Train: {len(train_dataset)}, Eval: {len(eval_dataset)} -- ready!')
        cache_loaded = True
    except Exception as e:
        print(f'[WARNING] Cache corrupted ({e}). Deleting and rebuilding...')
        shutil.rmtree(TRAIN_CACHE, ignore_errors=True)
        shutil.rmtree(EVAL_CACHE, ignore_errors=True)

# -------------------------------------------------------------------
# FIRST RUN: download + process + save to Drive cache
# -------------------------------------------------------------------
if not cache_loaded:
    print('[FIRST RUN] Downloading Im2LaTeX dataset...')
    try:
        train_data = load_dataset('yuntian-deng/im2latex-100k', split='train[:100000]')
        eval_data  = load_dataset('yuntian-deng/im2latex-100k', split='val[:5000]')
        print(f'[OK] Downloaded: {len(train_data)} train, {len(eval_data)} eval')
    except Exception as e:
        print(f'[ERROR] {e}')
        raise

    # num_proc=1 required on Colab: PIL images cause forked workers to hang at 0%
    print('[INFO] Preprocessing... (this takes ~10-15 min for 55k samples, one-time only)')
    train_dataset = train_data.map(
        preprocess_data,
        remove_columns=['image', 'formula'],
        batched=True,
        batch_size=64,
        num_proc=1,
        desc='Train'
    )
    eval_dataset = eval_data.map(
        preprocess_data,
        remove_columns=['image', 'formula'],
        batched=True,
        batch_size=64,
        num_proc=1,
        desc='Eval'
    )

    print('[INFO] Saving to Drive cache for future re-runs...')
    train_dataset.save_to_disk(TRAIN_CACHE)
    eval_dataset.save_to_disk(EVAL_CACHE)
    print(f'[OK] Cached! Train: {len(train_dataset)}, Eval: {len(eval_dataset)}')
    print('     Next run will load instantly from cache.')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

The image processor of type `DeiTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

...

Eval:   0%|          | 0/5000 [00:00<?, ? examples/s]

[INFO] Saving to Drive cache for future re-runs...


Saving the dataset (0/196 shards):   0%|          | 0/55033 [00:00<?, ? examples/s]

Saving the dataset (0/18 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

[OK] Cached! Train: 55033, Eval: 5000
     Next run will load instantly from cache.


In [3]:

import shutil, os

# Clear HuggingFace intermediate map() cache (the big one)
hf_cache = os.path.expanduser('~/.cache/huggingface')
if os.path.exists(hf_cache):
    size = sum(f.stat().st_size for f in __import__('pathlib').Path(hf_cache).rglob('*') if f.is_file())
    shutil.rmtree(hf_cache)
    print(f'[OK] Freed {size/1e9:.1f} GB from ~/.cache/huggingface/')

# Check disk now
import shutil as sh
total, used, free = sh.disk_usage('/')
print(f'Disk: {free/1e9:.1f} GB free of {total/1e9:.1f} GB')


[OK] Freed 107.6 GB from ~/.cache/huggingface/
Disk: 74.4 GB free of 253.1 GB


In [4]:
# ==============================================================================
# CELL 3 — Initialize TrOCR Model
# ==============================================================================
from transformers import VisionEncoderDecoderModel

print('[INFO] Loading TrOCR model...')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-small-printed')

# Configure decoding tokens for LaTeX
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size
model.config.eos_token_id           = processor.tokenizer.sep_token_id

# Setup generation_config (required by newer transformers versions)
model.generation_config.max_length             = 64
model.generation_config.early_stopping         = True
model.generation_config.num_beams              = 4
model.generation_config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.generation_config.pad_token_id           = processor.tokenizer.pad_token_id
model.generation_config.eos_token_id           = processor.tokenizer.sep_token_id

# Remove from config if they exist to prevent ValueError during save
for param in ['max_length', 'early_stopping', 'num_beams']:
    if hasattr(model.config, param):
        delattr(model.config, param)

print('[OK] Model ready.')
print(f'     Vocab size : {model.config.vocab_size}')
print(f'     Max length : {model.generation_config.max_length}')
print(f'     Num beams  : {model.generation_config.num_beams}')


[INFO] Loading TrOCR model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/246M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-small-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[OK] Model ready.
     Vocab size : 64044
     Max length : 64
     Num beams  : 4


In [5]:
# ==============================================================================
# CELL 4 — Training
# ==============================================================================
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers.trainer_utils import get_last_checkpoint
import torch


training_args = Seq2SeqTrainingArguments(
    output_dir                  = CHECKPOINT_DIR,    # saves to Drive

    # ── Generation ────────────────────────────────────────────────────────────
    predict_with_generate       = True,
    generation_max_length       = 64,

    # ── Batch size:
    per_device_train_batch_size = 96,
    per_device_eval_batch_size  = 96,
    gradient_accumulation_steps = 1,      # effective batch = 96 per step

    # ── Precision: ─────────
    bf16                        = True,
    fp16                        = False,

    # ── Data loading ───────────
    dataloader_num_workers      = 8,
    dataloader_pin_memory       = True,   # faster CPU->GPU transfer

    # ── Optimizer: fused AdamW ─────────────────────────
    optim                       = 'adamw_torch_fused',

    # ── Logging & Saving ─────────────────────────────────────────────────────
    logging_steps               = 50,
    save_steps                  = 200,    # save more often with bigger batches
    eval_steps                  = 200,
    save_total_limit            = 3,
    num_train_epochs            = 5,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,

    # ── Speed ─────────────────────────────────────────────────────────────────
    eval_strategy               = 'steps',
)

trainer = Seq2SeqTrainer(
    model            = model,
    processing_class = processor,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = eval_dataset,
)

# Auto-resume from last checkpoint (picks up where interrupted)
last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

try:
    if last_checkpoint:
        print(f'[RESUME] Resuming from {last_checkpoint}')
        print(f'         Batch size: 96, Grad Acc: 1 (maximizing L4 24GB VRAM)')
        try:
            trainer.train(resume_from_checkpoint=last_checkpoint)
        except ValueError as e:
            if "Can't find a valid checkpoint" in str(e):
                print(f"[WARNING] Invalid checkpoint at {last_checkpoint}. Falling back to fresh training...")
                trainer.train()
            else:
                raise
    else:
        print('[START] Starting fresh training with A100-optimized settings...')
        trainer.train()
    print('[OK] Training complete!')
except KeyboardInterrupt:
    print('[PAUSED] Re-run this cell to resume from last checkpoint.')
except Exception as e:
    print(f'[ERROR] {e}')
    raise


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 0}.


[RESUME] Resuming from /content/drive/MyDrive/Smart_Scan/Recognition_Model/trocr_checkpoints/checkpoint-2400
         Batch size: 96, Grad Acc: 1 (maximizing L4 24GB VRAM)


Step,Training Loss,Validation Loss
2600,0.218303,0.211728
2800,0.207225,0.208742


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[OK] Training complete!


In [6]:
# ==============================================================================
# CELL 5 — Save Final Model to Drive
# ==============================================================================
print('[INFO] Saving final model to Drive...')
trainer.save_model(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)
print(f'[OK] Model saved -> {FINAL_MODEL_DIR}')


[INFO] Saving final model to Drive...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[OK] Model saved -> /content/drive/MyDrive/Smart_Scan/Recognition_Model/trocr_final


In [7]:
# ==============================================================================
# CELL 6 — Verify Saved Files
# ==============================================================================
import os

final_files      = os.listdir(FINAL_MODEL_DIR) if os.path.exists(FINAL_MODEL_DIR) else []
checkpoint_files = os.listdir(CHECKPOINT_DIR)  if os.path.exists(CHECKPOINT_DIR)  else []

print(f'[Drive] Final model  : {FINAL_MODEL_DIR}')
print(f'        Files        : {final_files}')
print(f'[Drive] Checkpoints  : {CHECKPOINT_DIR}')
print(f'        Files (first5): {checkpoint_files[:5]}')

if final_files:
    print('[OK] Model saved successfully on Drive!')
else:
    print('[WARN] No files found in final model directory — run Cell 5 first.')


[Drive] Final model  : /content/drive/MyDrive/Smart_Scan/Recognition_Model/trocr_final
        Files        : ['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer.json', 'processor_config.json', 'training_args.bin', 'tokenizer_config.json']
[Drive] Checkpoints  : /content/drive/MyDrive/Smart_Scan/Recognition_Model/trocr_checkpoints
        Files (first5): ['checkpoint-2600', 'checkpoint-2800', 'checkpoint-2870']
[OK] Model saved successfully on Drive!


In [9]:
!pip install evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 100.1 MB/s eta 0:00:00


In [10]:
# ==============================================================================
# CELL 7 — Evaluate Accuracy & Performance
# ==============================================================================
import torch
from tqdm.auto import tqdm
import evaluate

print('[INFO] Running standard evaluation on validation set...')
try:
    eval_metrics = trainer.evaluate()
    print(f"Validation Loss: {eval_metrics.get('eval_loss', 'N/A'):.4f}")
except Exception as e:
    print(f"[WARN] Standard evaluation failed or trainer not loaded: {e}")

try:
    print('[INFO] Loading metrics (CER and BLEU)...')
    cer_metric = evaluate.load("cer")
    bleu_metric = evaluate.load("bleu")

    print('[INFO] Computing CER and BLEU on a subset of validation data...')
    model.eval()
    predictions = []
    references = []

    # Run over a subset (e.g., 100 samples) to evaluate quickly
    num_samples = min(100, len(eval_dataset))

    with torch.no_grad():
        for i in tqdm(range(num_samples)):
            sample = eval_dataset[i]
            # Handle pixel_values whether they are tensors or lists
            pixel_values = sample['pixel_values']
            if not isinstance(pixel_values, torch.Tensor):
                pixel_values = torch.tensor(pixel_values)

            pixel_values = pixel_values.unsqueeze(0).to(model.device)
            labels = sample['labels']
            if not isinstance(labels, torch.Tensor):
                labels = torch.tensor(labels)

            # Generate prediction
            generated_ids = model.generate(pixel_values, max_length=64, early_stopping=True, num_beams=4)
            pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

            # Decode reference
            labels[labels == -100] = processor.tokenizer.pad_token_id
            ref_text = processor.decode(labels, skip_special_tokens=True)

            predictions.append(pred_text.strip())
            references.append(ref_text.strip())

    cer_score = cer_metric.compute(predictions=predictions, references=references)
    bleu_score = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])

    print("\n" + "="*40)
    print("         PERFORMANCE METRICS")
    print("="*40)
    print(f"Character Error Rate (CER): {cer_score:.4f} (Lower is better)")
    print(f"BLEU Score                : {bleu_score['bleu']:.4f} (Higher is better)")
    print("="*40)

    print("\n--- Sample Predictions ---")
    for i in range(min(5, num_samples)):
        print(f"Target : {references[i]}")
        print(f"Predict: {predictions[i]}")
        print("-" * 40)

except ImportError:
    print('\n[ERROR] Missing required packages for CER/BLEU computation.')
    print('Please run this command in a new cell first:')
    print('!pip install evaluate jiwer')
except Exception as e:
    print(f"\n[ERROR] Evaluation encountered an issue: {e}")


[INFO] Running standard evaluation on validation set...


Validation Loss: 0.2087
[INFO] Loading metrics (CER and BLEU)...


[INFO] Computing CER and BLEU on a subset of validation data...


  0%|          | 0/100 [00:00<?, ?it/s]


         PERFORMANCE METRICS
Character Error Rate (CER): 0.0967 (Lower is better)
BLEU Score                : 0.8842 (Higher is better)

--- Sample Predictions ---
Target : E ( v ) = \frac { d } { d t } E ( q ) \; \; \; \; \; \; \; \forall t \, .
Predict: E ( v ) = \frac { d } { d t } E ( q ) \qquad \forall t \, .
----------------------------------------
Target : { \frac { 1 } { L ^ { 2 } } } \prod _ { i = 1 } ^ { 3 } ( r _ { + } ^ { 2 } + q _ { i } ) - \mu r _ { + } ^ { 2 } \sim 0 \ .
Predict: \frac { 1 } { L ^ { 2 } } \prod _ { i = 1 } ^ { 3 } ( r _ { + } ^ { 2 } + q _ { i } ) - \mu r _ { + } ^ { 2 } \sim 0 \ .
----------------------------------------
Target : x ^ { I } ( \sigma + 2 \pi , \tau ) = x ^ { I } ( \sigma , \tau ) + 2 \pi L ^ { I } .
Predict: x ^ { I } ( \sigma + 2 \pi , \tau ) = x ^ { I } ( \sigma , \tau ) + 2 \pi L ^ { I } .
----------------------------------------
Target : \xi _ { \sigma } ( \sigma ) = 1 - \frac { \pi e ^ { \sigma } } { 3 } + 8 \pi e ^ { \sigma } \sum 